In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ltn
!pip install numpy pandas matplotlib
!pip install torch
!pip install torchvision
!pip install torchmetrics

testes

In [8]:
import ltn
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

from PIL import Image
import os

import torch
from torch import optim
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from tqdm import tqdm

# !pip install torchvision
import torchvision

import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms

# !pip install torchmetrics
import torchmetrics

""" HYPERPARAMETERS """


DATA_ROOT = Path("/content/drive/MyDrive/datasetVCB")
BATCH_SIZE = 32
COMPLEXITY_DATASET = 1
EPOCHS = 50
TOTAL_EXAMPLES = 15000
TRAIN_TEST_LIMIT = 12000
LEARNING_RATE = 0.001
csv_path = "results.csv"
txt_path = "stats.txt"
confusion_matrix_path = "conf_matrix.txt"

In [ ]:
BATCH_SIZE = 32

In [9]:

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(DATA_ROOT, "all.csv")
images = Path(DATA_ROOT, "c1")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]
CLASSES = [
    "Cafe",
    "Hotel",
    "Store",
    "MiscCommercial",
    "Awning",
    "Billboard",
    "VendingMachine",
    "Statue",
    "Table",
    "WallSign",
]

class BuildingsDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load image
        #img_path = os.path.join(self.img_dir, row["id"])

        img_path = self.img_dir / row["id"]
        image = Image.open(img_path).convert("RGB")

        # Label (adjust depending on your task)
        #label = torch.tensor(row["label"], dtype=torch.long)
        # OR for multi-label:
        target = torch.tensor(row[CLASSES].values, dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, target


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


TRAIN_TEST_LIMIT = int(0.8 * len(df))

train_df = df[:TRAIN_TEST_LIMIT]
test_df = df[TRAIN_TEST_LIMIT:]

train_dataset = BuildingsDataset(train_df, images, transform=transform)
test_dataset = BuildingsDataset(test_df, images, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

CNN model

In [12]:
#print(train_df["Cafe"])

"""CNN MODEL"""
class CNN(nn.Module):
   def __init__(self, in_channels, num_classes, name="cnn_model"):

       """
       Building blocks of convolutional neural network.

       Parameters:
           * in_channels: Number of channels in the input image (for grayscale images, 1)
           * num_classes: Number of classes to predict. In our problem, 10 (i.e digits from  0 to 9).
       """
       super(CNN, self).__init__(name=name)

       #layers
       self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)
       self.conv2 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3)
       self.conv3 = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3)

       self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
       self.flatten = nn.Flatten()
       self.dropout = nn.Dropout(0.5)


       # 2nd convolutional layer
       self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1)
       # Fully connected layer
       self.fc1 = nn.Linear(16 * 7 * 7, num_classes)

   def forward(self, x):
       """
       Define the forward pass of the neural network.

       Parameters:
           x: Input tensor.

       Returns:
           torch.Tensor
               The output tensor after passing through the network.
       """
       x = F.relu(self.conv1(x))  # Apply first convolution and ReLU activation
       x = self.pool(x)           # Apply max pooling
       x = F.relu(self.conv2(x))  # Apply second convolution and ReLU activation
       x = self.pool(x)           # Apply max pooling
       x = x.reshape(x.shape[0], -1)  # Flatten the tensor
       x = self.fc1(x)            # Apply fully connected layer
       return x
       x = x.reshape(x.shape[0], -1)  # Flatten the tensor
       x = self.fc1(x)            # Apply fully connected layer
       return x




class CNNModel(tf.keras.Model):

    def __init__(self, num_classes, name="cnn_model"):
        super(CNNModel, self).__init__(name="cnn_model")
        self.conv1 = keras.layers.Conv2D(
            16, kernel_size=(3, 3), activation="relu", input_shape=(224, 224, 3)
        )
        self.conv2 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")
        self.conv3 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu")

        self.maxpool = keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.5)
        self.dense1 = keras.layers.Dense(128, activation="relu")
        self.dense2 = keras.layers.Dense(64, activation="relu")
        self.concept_layer = keras.layers.Dense(32, activation="relu")
        self.output_layer = keras.layers.Dense(num_classes)

    def call(self, inputs):
        if isinstance(inputs, (list, tuple)):
          inputs = inputs[0]

        x = self.conv1(inputs) #thinha um 0
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.dense1(x)
        x = self.dropout(x)
        x = self.dense2(x)
        y = self.concept_layer(x)
        x = self.output_layer(y)
        result = tf.concat([x, y], axis=1)
        return result


0        0
1        0
2        0
3        1
4        0
        ..
15995    0
15996    1
15997    0
15998    0
15999    1
Name: Cafe, Length: 16000, dtype: int64
